# Advanced: Multi-Strategy Dataset Splitting with PAMOLA.CORE

**Goal**: Master all 3 dataset splitting strategies using operation.execute()

**What you'll learn:**
- **Strategy 1**: Value Groups - Explicit ID-based splitting with custom groups
- **Strategy 2**: Equal Size - Automatic partitioning into balanced subsets
- **Strategy 3**: Modulo-Based - Hash-based distribution for scalable splitting
- Compare performance and distribution across strategies
- Analyze record distribution and partition balance
- Production deployment patterns for large datasets

**Prerequisites:**
- Completed the simple notebook
- Understanding of data partitioning concepts
- Familiarity with operation.execute() pattern
- Python 3.8+
- PAMOLA.CORE installed (auto-installs if needed)

---

## Step 1: Setup and Imports

**How to use:**
- Run this cell to import required libraries
- Auto-detects PAMOLA installation path (searches up 6 levels)
- If not found locally, auto-installs from GitHub

**What you'll see:**
- Detection message showing PAMOLA location or installation progress
- Import confirmation (✅ All imports successful!)
- Notebook start timestamp
- Project root and working directory paths

**Note:** First run may take 1-2 minutes if installing from GitHub

**Expected structure:**
```
PAMOLA/
├── pamola_core/          ← Auto-detected
└── examples/
    ├── data_examples/    ← Data directory
    └── transformations/
        └── 02_split_by_id_values_advanced.ipynb  ← You are here
```

In [ ]:
import pandas as pd
import numpy as np
import json
import sys
import os
from pathlib import Path
from datetime import datetime
import time
from IPython.display import Image, display

print("🔍 Detecting PAMOLA installation...\n")

# Auto-detect project root (search up 6 levels for pamola_core/)
current_dir = Path.cwd()
project_root = current_dir
pamola_found = False

for level in range(6):
    if (project_root / 'pamola_core').exists():
        print(f"✓ Found pamola_core at level {level}: {project_root}")
        sys.path.insert(0, str(project_root))
        pamola_found = True
        break
    project_root = project_root.parent

# If not found locally, try to install from Git
if not pamola_found:
    print("⚠️  pamola_core not found in project structure")
    print("📦 Attempting to install from GitHub...\n")
    
    try:
        import subprocess
        
        # Install from Git repository
        install_cmd = [
            sys.executable, "-m", "pip", "install",
            "git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        ]
        
        result = subprocess.run(
            install_cmd,
            capture_output=True,
            text=True,
            check=True
        )
        
        print("✅ Successfully installed pamola_core from GitHub!")
        print("   Repository: https://github.com/DGT-Network/PAMOLA.git")
        print("   Branch: pre-epic3")
        
    except subprocess.CalledProcessError as e:
        print(f"❌ Installation failed!")
        print(f"   Error: {e.stderr}")
        raise RuntimeError(
            "Could not find pamola_core locally and installation from GitHub failed. "
            "Please install manually using:\n"
            "pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        )
    except Exception as e:
        print(f"❌ Unexpected error during installation: {e}")
        raise

# Now try to import the required modules
try:
    from pamola_core.transformations.splitting.split_by_id_values_op import SplitByIDValuesOperation
    from pamola_core.utils.ops.op_data_source import DataSource
    from pamola_core.utils.progress import HierarchicalProgressTracker
    from pamola_core.utils.tasks.task_reporting import TaskReporter
    from pamola_core.io.csv import read_csv
    
    print("✅ All imports successful!")
    print(f"📅 Notebook started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 80)
    print(f"📂 Project root:  {project_root.name}")
    print(f"📂 Working dir:   {current_dir.relative_to(project_root)}")
    print("=" * 80)
    
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("\n💡 Troubleshooting:")
    print("   1. Try restarting your Python kernel/notebook")
    print("   2. Verify installation: pip list | grep pamola")
    print("   3. Manual install: pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3")
    raise

## Step 2: Load Complex Dataset

**How to use:**
- Run to load or generate 1000-record dataset
- Auto-creates sample if file not found
- Review data structure before proceeding

**What you'll see:**
- Load status (from file or generated)
- Dataset overview (1000 records, 8 columns)
- Sample records (first 10 rows)
- Column statistics (types, ranges, unique counts)

**Dataset features:**
- 1000 transaction records
- 25 unique customer IDs (C001-C025)
- Realistic transaction data with dates, products, amounts
- Varied distribution for testing different strategies

**Note:** Generated data auto-saves to examples/data_examples/ for reuse

In [ ]:
# Try to load from file first (in examples directory)
examples_dir = project_root / 'examples'
data_path = examples_dir / 'data_examples' / 'split_complex_data.csv'
print(f"📂 Looking for data at: {data_path}\n")

if data_path.exists():
    print(f"📂 Loading data from: {data_path}")
    df = read_csv(data_path)
    print(f"✓ Loaded {len(df)} records from file")
else:
    print("📊 Generating synthetic dataset (1000 records)...\n")
    
    np.random.seed(42)
    n_records = 1000
    
    # Generate 25 customer IDs for realistic distribution
    customer_ids = [f'C{i:03d}' for i in range(1, 26)]
    
    # Product categories
    products = [
        'Laptop', 'Desktop', 'Phone', 'Tablet', 'Monitor', 
        'Keyboard', 'Mouse', 'Printer', 'Scanner', 'Webcam',
        'Headphones', 'Speaker', 'Router', 'Switch', 'Cable'
    ]
    
    # Payment methods
    payment_methods = ['Credit Card', 'Debit Card', 'PayPal', 'Cash', 'Bank Transfer']
    
    # Generate transactions with realistic distribution
    df = pd.DataFrame({
        'transaction_id': range(1, n_records + 1),
        'customer_id': np.random.choice(customer_ids, n_records),
        'transaction_date': pd.date_range('2024-01-01', periods=n_records, freq='8h'),
        'product': np.random.choice(products, n_records),
        'quantity': np.random.randint(1, 10, n_records),
        'amount': np.random.randint(50, 5000, n_records),
        'payment_method': np.random.choice(payment_methods, n_records),
        'status': np.random.choice(['Completed', 'Pending', 'Cancelled'], n_records, p=[0.8, 0.15, 0.05])
    })
    
    # Save for future use (with error handling)
    try:
        data_path.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(data_path, index=False)
        print(f"✓ Generated and saved to: {data_path}")
    except PermissionError:
        print(f"⚠️  Cannot save to {data_path} (file may be open)")
        print(f"   Dataset generated in memory only")

print(f"\n📊 Dataset Overview:")
print("=" * 80)
print(f"  Records: {len(df):,}")
print(f"  Columns: {', '.join(df.columns)}")

print(f"\n🔍 Sample Records (first 10):")
print(df.head(10))

print(f"\n📈 Column Statistics:")
for col in df.columns:
    dtype_str = str(df[col].dtype)
    if pd.api.types.is_string_dtype(df[col]):
        print(f"  {col:20s} ({dtype_str:10s}): {df[col].nunique()} unique values")
    else:
        non_null = df[col].dropna()
        if len(non_null) > 0:
            print(f"  {col:20s} ({dtype_str:10s}): min={non_null.min()}, max={non_null.max()}")

print(f"\n💡 Perfect dataset for testing all 3 splitting strategies!")

## Step 3: Setup Shared Environment

**How to use:**
1. **CUSTOMIZE FIELD_CONFIG**: Edit id_field for all strategies
   - `"id_field": "customer_id"` - Change to YOUR dataset's ID column
2. Run to validate field and setup environment

**What you'll see:**
- Field validation (✓ id_field exists, unique value count)
- Sample ID values from dataset
- Task directory created (✓)
- TaskReporter initialized (✓)
- DataSource and config ready (✓)

**Configuration:**
```python
FIELD_CONFIG = {
    "id_field": "customer_id"  # CUSTOMIZE: Your ID column
}
```

**Note:** ID field must exist in dataset. All strategies will use this field for splitting

In [ ]:
# Field configuration - CUSTOMIZE THIS!
FIELD_CONFIG = {
    "id_field": "customer_id"  # Field used for splitting in all strategies
}

# Validate field exists in dataset
print("📋 Validating field configuration...\n")
id_field = FIELD_CONFIG["id_field"]

if id_field not in df.columns:
    raise ValueError(
        f"❌ Field '{id_field}' not found!\n"
        f"Available columns: {', '.join(df.columns)}\n"
        f"Please update FIELD_CONFIG"
    )

print(f"  ✓ ID field: '{id_field}'")
print(f"    Unique values: {df[id_field].nunique()}")
print(f"    Sample IDs: {list(df[id_field].unique()[:5])}")

# Configuration helper (production pattern)
def create_config_kwargs():
    return {
        "dataset_name": "main_dataset"
    }

# Create task directory (in examples/data_examples)
examples_dir = project_root / 'examples'
task_dir = examples_dir / 'data_examples' / 'advanced_output'
os.makedirs(task_dir, exist_ok=True)
print(f"\n✓ Task directory: {task_dir}")

# Initialize TaskReporter
task_reporter = TaskReporter(
    task_id="advanced_001",
    task_type="multi_strategy_split",
    description="Multi-strategy dataset splitting",
    report_path=task_dir
)
print("✓ TaskReporter initialized")

# Get config
kwargs = create_config_kwargs()
print(f"✓ Config kwargs ready")

# Create DataSource
data_source = DataSource(
    dataframes={"main_dataset": df}
)
print("✓ DataSource created")

print(f"\n✅ Shared environment ready for all strategies!")

## STRATEGY 1: Value Groups (Explicit Splitting)

**How to use:**
- Review pre-configured customer groups
- Run to execute explicit value-based splitting
- Monitor progress bar (6 steps)

**Key parameters:**
- `value_groups`: Dictionary defining explicit groups
  - `premium_customers`: C001-C005 (high-value)
  - `regular_customers`: C006-C015 (medium-value)
  - `new_customers`: C016-C020 (recent)
- Unmatched IDs automatically go to 'others' group

**What you'll see:**
- Configuration summary with 3 defined groups
- Progress: validation → filtering → saving → metrics → viz
- Completion time (1-3 seconds)
- Output: 4 files (3 groups + 'others')

**Validate:**
- ✅ Execution time <5 seconds
- ✅ 4 output files created (premium, regular, new, others)
- ✅ Total records = original dataset
- ✅ Status = "completed"

**Note:** Best for business-driven segmentation with known customer groups

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 1: VALUE GROUPS (EXPLICIT SPLITTING)")
print("=" * 80 + "\n")

# Setup progress tracker
tracker_s1 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 1: Value Groups",
    unit="steps",
    track_memory=True,
    level=0
)

# Define explicit customer groups
value_groups_s1 = {
    "premium_customers": ['C001', 'C002', 'C003', 'C004', 'C005'],
    "regular_customers": ['C006', 'C007', 'C008', 'C009', 'C010', 
                          'C011', 'C012', 'C013', 'C014', 'C015'],
    "new_customers": ['C016', 'C017', 'C018', 'C019', 'C020']
}

# Create operation
operation_s1 = SplitByIDValuesOperation(
    # Core parameters
    id_field=id_field,
    value_groups=value_groups_s1,
    
    # Output configuration
    output_format='csv',
    
    # Processing settings
    use_vectorization=False,
    parallel_processes=1,
    use_cache=False,
    
    # Execution behavior
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 1 configured")
print(f"  Value groups: {len(value_groups_s1)}")
for group_name, ids in value_groups_s1.items():
    print(f"    {group_name}: {len(ids)} IDs")

print(f"\n⚙️  Executing...\n")
start_time = time.time()

# Execute
result_s1 = operation_s1.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy1_value_groups',
    reporter=task_reporter,
    progress_tracker=tracker_s1,
    **kwargs
)

elapsed_s1 = time.time() - start_time
print(f"\n✅ Strategy 1 completed in {elapsed_s1:.2f}s")

# Count output files
output_dir_s1 = task_dir / 'strategy1_value_groups' / 'output'
if output_dir_s1.exists():
    output_files_s1 = list(output_dir_s1.glob('*.csv'))
    print(f"\n📊 Generated {len(output_files_s1)} output files")
    
    # Show file sizes
    for f in sorted(output_files_s1, key=lambda x: x.stat().st_mtime, reverse=True):
        size_kb = f.stat().st_size / 1024
        # Extract group name from filename
        parts = f.stem.split('_')
        try:
            output_idx = parts.index('output')
            group_name = '_'.join(parts[5:output_idx])
        except (ValueError, IndexError):
            group_name = f.stem
        print(f"  • {group_name}: {size_kb:.1f} KB")

## STRATEGY 2: Equal Size Partitioning

**How to use:**
- Automatically splits into 3 equal-sized partitions
- No manual group definition needed
- Run to execute balanced partitioning

**Key parameters:**
- `number_of_partitions=3`: Create 3 balanced subsets
- `partition_method='equal_size'`: Balance by record count
- Algorithm: Sorts by ID, then divides into equal chunks

**What you'll see:**
- Configuration confirmation
- Progress: validation → sorting → partitioning → metrics → viz
- Completion time (1-3 seconds)
- Output: 3 files with similar record counts (~333 each)

**Validate:**
- ✅ Execution time <5 seconds
- ✅ 3 output files (partition_0, partition_1, partition_2)
- ✅ Balanced sizes (max difference <2 records)
- ✅ Status = "completed"

**Note:** Best for load balancing and parallel processing scenarios

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 2: EQUAL SIZE PARTITIONING")
print("=" * 80 + "\n")

tracker_s2 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 2: Equal Size",
    unit="steps",
    track_memory=True,
    level=0
)

operation_s2 = SplitByIDValuesOperation(
    # Core parameters
    id_field=id_field,
    number_of_partitions=3,
    partition_method='equal_size',
    
    # Output configuration
    output_format='csv',
    
    # Processing settings
    use_vectorization=False,
    parallel_processes=1,
    use_cache=False,
    
    # Execution behavior
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 2 configured")
print(f"  Partitions: {operation_s2.number_of_partitions}")
print(f"  Method: {operation_s2.partition_method}")
print(f"  Expected size per partition: ~{len(df) // 3} records")

print(f"\n⚙️  Executing...\n")
start_time = time.time()

result_s2 = operation_s2.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy2_equal_size',
    reporter=task_reporter,
    progress_tracker=tracker_s2,
    **kwargs
)

elapsed_s2 = time.time() - start_time
print(f"\n✅ Strategy 2 completed in {elapsed_s2:.2f}s")

# Count output files
output_dir_s2 = task_dir / 'strategy2_equal_size' / 'output'
if output_dir_s2.exists():
    output_files_s2 = list(output_dir_s2.glob('*.csv'))
    print(f"\n📊 Generated {len(output_files_s2)} partitions")
    
    # Show partition sizes
    for f in sorted(output_files_s2, key=lambda x: x.name):
        part_df = pd.read_csv(f)
        parts = f.stem.split('_')
        try:
            output_idx = parts.index('output')
            partition_name = '_'.join(parts[5:output_idx])
        except (ValueError, IndexError):
            partition_name = f.stem
        print(f"  • {partition_name}: {len(part_df)} records")

## STRATEGY 3: Modulo-Based Partitioning

**How to use:**
- Hash-based distribution using modulo operation
- Deterministic and reproducible splitting
- Run to execute scalable hash-based partitioning

**Key parameters:**
- `number_of_partitions=4`: Create 4 hash-based partitions
- `partition_method='modulo'`: Use hash(id) % 4
- Algorithm: Consistent hashing for stable distribution

**What you'll see:**
- Configuration confirmation
- Progress: validation → hashing → distribution → metrics → viz
- Completion time (1-3 seconds)
- Output: 4 files with roughly equal sizes (~250 each)

**Validate:**
- ✅ Execution time <5 seconds
- ✅ 4 output files (partition_0 through partition_3)
- ✅ Deterministic distribution (same ID always goes to same partition)
- ✅ Status = "completed"

**Note:** Best for distributed systems, sharding, and horizontal scaling

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 3: MODULO-BASED PARTITIONING")
print("=" * 80 + "\n")

tracker_s3 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 3: Modulo",
    unit="steps",
    track_memory=True,
    level=0
)

operation_s3 = SplitByIDValuesOperation(
    # Core parameters
    id_field=id_field,
    number_of_partitions=4,
    partition_method='modulo',
    
    # Output configuration
    output_format='csv',
    
    # Processing settings
    use_vectorization=False,
    parallel_processes=1,
    use_cache=False,
    
    # Execution behavior
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 3 configured")
print(f"  Partitions: {operation_s3.number_of_partitions}")
print(f"  Method: {operation_s3.partition_method}")
print(f"  Expected size per partition: ~{len(df) // 4} records")

print(f"\n⚙️  Executing...\n")
start_time = time.time()

result_s3 = operation_s3.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy3_modulo',
    reporter=task_reporter,
    progress_tracker=tracker_s3,
    **kwargs
)

elapsed_s3 = time.time() - start_time
print(f"\n✅ Strategy 3 completed in {elapsed_s3:.2f}s")

# Count output files
output_dir_s3 = task_dir / 'strategy3_modulo' / 'output'
if output_dir_s3.exists():
    output_files_s3 = list(output_dir_s3.glob('*.csv'))
    print(f"\n📊 Generated {len(output_files_s3)} partitions")
    
    # Show partition sizes and sample IDs
    for f in sorted(output_files_s3, key=lambda x: x.name):
        part_df = pd.read_csv(f)
        parts = f.stem.split('_')
        try:
            output_idx = parts.index('output')
            partition_name = '_'.join(parts[5:output_idx])
        except (ValueError, IndexError):
            partition_name = f.stem
        unique_ids = part_df[id_field].nunique()
        print(f"  • {partition_name}: {len(part_df)} records, {unique_ids} unique IDs")

## Step 4: Strategy Comparison

**How to use:**
- Run AFTER all 3 strategies complete
- Review execution times and distribution metrics
- Identify best strategy for your use case

**What you'll see:**
- Execution time comparison (fastest to slowest)
- Total processing time
- Partition count comparison
- Distribution balance analysis

**Strategy selection guide:**
- **Value Groups**: Known business segments, explicit control
- **Equal Size**: Load balancing, parallel processing
- **Modulo**: Distributed systems, horizontal scaling, sharding

**Note:** Fastest strategy isn't always best - consider your specific requirements

In [ ]:
print("\n" + "=" * 80)
print("📊 STRATEGY COMPARISON")
print("=" * 80 + "\n")

print("⏱️  Execution Time:")
print(f"  Strategy 1 (Value Groups): {elapsed_s1:6.2f}s")
print(f"  Strategy 2 (Equal Size):   {elapsed_s2:6.2f}s")
print(f"  Strategy 3 (Modulo):       {elapsed_s3:6.2f}s")
print(f"  Total:                     {elapsed_s1 + elapsed_s2 + elapsed_s3:6.2f}s")

# Compare partition counts
print("\n📈 Partition Counts:")
if output_dir_s1.exists():
    s1_count = len(list(output_dir_s1.glob('*.csv')))
    print(f"  Strategy 1: {s1_count} groups (3 defined + others)")
if output_dir_s2.exists():
    s2_count = len(list(output_dir_s2.glob('*.csv')))
    print(f"  Strategy 2: {s2_count} partitions (equal size)")
if output_dir_s3.exists():
    s3_count = len(list(output_dir_s3.glob('*.csv')))
    print(f"  Strategy 3: {s3_count} partitions (hash-based)")

print("\n🎯 Key Insights:")
print(f"  • Total dataset: {len(df):,} records")
print(f"  • Unique IDs: {df[id_field].nunique()}")
print(f"  • All strategies preserve total record count")
print(f"  • Distribution varies by strategy design")

## Step 5: Detailed Artifact Review

**How to use:**
- Review all generated artifacts from each strategy
- Auto-loads NEWEST files from each category
- Displays metrics and sample data inline

**What you'll see (per strategy):**
1. **Metrics JSON**: Split statistics, record counts, execution time
2. **Output Files**: Split datasets with record counts per partition
3. **Visualizations**: Distribution charts (latest batch only)

**Validate:**
- ✅ All metrics show expected partition counts
- ✅ Total output records = input records (1000)
- ✅ File sizes appropriate for record counts
- ✅ Visualizations show clear distribution

**Note:** Only newest files shown. Multiple runs create versions - older artifacts excluded automatically

In [ ]:
# Review all strategies
strategy_dirs = [
    ('strategy1_value_groups', 'Strategy 1: Value Groups'),
    ('strategy2_equal_size', 'Strategy 2: Equal Size'),
    ('strategy3_modulo', 'Strategy 3: Modulo-Based')
]

for dir_name, title in strategy_dirs:
    strategy_dir = task_dir / dir_name
    if not strategy_dir.exists():
        continue
        
    print("\n" + "=" * 80)
    print(f"📊 {title}")
    print("=" * 80)
    
    # 1. Metrics (NEWEST - with filtering)
    metrics_dir = strategy_dir / 'metrics'
    if metrics_dir.exists():
        metrics_files = list(metrics_dir.glob('*.json'))
        if metrics_files:
            # Exclude data_types files
            filtered = [f for f in metrics_files if not f.name.startswith("data_types")]

            if filtered:
                target_files = filtered
                print(f"✓ Found {len(filtered)} metrics file(s)")
            else:
                target_files = metrics_files
                print(f"⚠ Fallback to ALL {len(metrics_files)} file(s)")

            # Pick latest
            target_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
            latest_metrics_file = target_files[0]
            
            print(f"📄 {latest_metrics_file.name}")
            
            try:
                with open(latest_metrics_file, 'r') as f:
                    data = json.load(f)
                    metrics = data if isinstance(data, dict) else data.get('metrics', {})
                
                # Display key metrics
                print(f"   Input records: {metrics.get('total_input_records', 'N/A')}")
                print(f"   Output records: {metrics.get('total_output_records', 'N/A')}")
                print(f"   Number of splits: {metrics.get('number_of_splits', 'N/A')}")
                print(f"   Execution time: {metrics.get('execution_time_seconds', 0):.4f}s")
                
                # Show split info if available
                if 'split_info' in metrics:
                    print(f"\n   Split Details:")
                    for split_name, info in list(metrics['split_info'].items())[:5]:
                        print(f"     • {split_name}: {info.get('record_count', 0)} records")
                    
            except Exception as e:
                print(f"   ⚠️  Error: {e}")
    
    # 2. Output Files (NEWEST BATCH)
    output_dir = strategy_dir / 'output'
    if output_dir.exists():
        output_files = sorted(
            list(output_dir.glob('*.csv')),
            key=lambda x: x.stat().st_mtime, reverse=True
        )
        
        if output_files:
            latest_time = output_files[0].stat().st_mtime
            latest_batch = [
                f for f in output_files 
                if abs(f.stat().st_mtime - latest_time) < 10
            ]
            
            print(f"\n📁 OUTPUT FILES: {len(latest_batch)} files")
            for f in sorted(latest_batch, key=lambda x: x.name)[:5]:
                try:
                    part_df = pd.read_csv(f)
                    parts = f.stem.split('_')
                    try:
                        output_idx = parts.index('output')
                        group_name = '_'.join(parts[5:output_idx])
                    except (ValueError, IndexError):
                        group_name = f.stem
                    print(f"   • {group_name}: {len(part_df)} records")
                except Exception as e:
                    print(f"   ⚠️  Error reading {f.name}")
    
    # 3. Visualizations (NEWEST BATCH)
    viz_dir = strategy_dir / 'visualizations'
    if viz_dir.exists():
        viz_files = sorted(
            list(viz_dir.glob('*.png')),
            key=lambda x: x.stat().st_mtime, reverse=True
        )
        
        if viz_files:
            latest_time = viz_files[0].stat().st_mtime
            latest_batch = [
                f for f in viz_files 
                if abs(f.stat().st_mtime - latest_time) < 10
            ]
            
            print(f"\n📊 VISUALIZATIONS: {len(latest_batch)} files")
            for viz in latest_batch[:2]:  # Show first 2
                print(f"   📈 {viz.stem.replace('_', ' ').title()}")
                try:
                    display(Image(filename=str(viz)))
                except:
                    print(f"      (Display error)")

## Step 6: Export Final Data

**How to use:**
- Run AFTER all strategies complete
- Exports split datasets from all strategies
- Organizes files by strategy for easy access

**What you'll see (per strategy):**
- File location paths
- Record count per split
- Preview of sample records (first 3 rows)
- Summary statistics

**Output structure:**
```
advanced_output/
├── strategy1_value_groups/
│   └── output/
│       ├── ...premium_customers....csv
│       ├── ...regular_customers....csv
│       ├── ...new_customers....csv
│       └── ...others....csv
├── strategy2_equal_size/
│   └── output/
│       ├── ...partition_0....csv
│       ├── ...partition_1....csv
│       └── ...partition_2....csv
└── strategy3_modulo/
    └── output/
        ├── ...partition_0....csv
        ├── ...partition_1....csv
        ├── ...partition_2....csv
        └── ...partition_3....csv
```

**Note:** Files already saved during execution - this cell provides summary and verification

In [ ]:
print("=" * 80)
print("📦 EXPORT SUMMARY - ALL STRATEGIES")
print("=" * 80)

print(f"\n📂 Base directory: {task_dir}\n")

total_files = 0
total_records = 0

for dir_name, title in strategy_dirs:
    strategy_dir = task_dir / dir_name
    output_dir = strategy_dir / 'output'
    
    if not output_dir.exists():
        continue
    
    print("=" * 80)
    print(f"📊 {title}")
    print("=" * 80)
    
    output_files = list(output_dir.glob('*.csv'))
    strategy_records = 0
    
    print(f"\n📁 Location: {output_dir}")
    print(f"📄 Files: {len(output_files)}\n")
    
    # Show each file
    for f in sorted(output_files, key=lambda x: x.name):
        try:
            split_df = pd.read_csv(f)
            
            # Extract split name
            parts = f.stem.split('_')
            try:
                output_idx = parts.index('output')
                split_name = '_'.join(parts[5:output_idx])
            except (ValueError, IndexError):
                split_name = f.stem
            
            print(f"  {split_name}")
            print(f"    Records: {len(split_df)}")
            print(f"    Unique {id_field}s: {split_df[id_field].nunique()}")
            
            # Show first 3 rows
            print(f"\n    Sample (first 3):")
            display_cols = ['transaction_id', id_field, 'product', 'amount', 'status']
            display_cols = [col for col in display_cols if col in split_df.columns]
            print(split_df[display_cols].head(3).to_string(index=False, max_colwidth=15))
            print()
            
            strategy_records += len(split_df)
            
        except Exception as e:
            print(f"  ⚠️  Error reading {f.name}: {e}")
    
    print(f"  Strategy Total: {strategy_records} records")
    total_files += len(output_files)
    total_records += strategy_records

# ============================================================================
# GRAND SUMMARY
# ============================================================================
print("\n" + "=" * 80)
print("✅ EXPORT COMPLETE")
print("=" * 80)
print(f"\n📊 Grand Total:")
print(f"   Strategies executed: 3")
print(f"   Total files created: {total_files}")
print(f"   Total records (across all files): {total_records}")
print(f"   Original dataset: {len(df)} records")
print(f"   Verification: {'✅ PASS' if total_records == len(df) * 3 else '⚠️  MISMATCH'}")

if 'elapsed_s1' in locals() and 'elapsed_s2' in locals() and 'elapsed_s3' in locals():
    total_time = elapsed_s1 + elapsed_s2 + elapsed_s3
    print(f"\n⏱️  Total execution time: {total_time:.2f}s")
    print(f"   Average per strategy: {total_time/3:.2f}s")

print("\n" + "=" * 80)

## 🎯 Summary

**Accomplished:**
- ✅ 3 splitting strategies implemented and compared
- ✅ Performance benchmarking completed
- ✅ Distribution analysis across strategies
- ✅ Production-ready artifacts generated

**Key takeaways:**
- **Value Groups** provides explicit control over segmentation
- **Equal Size** ensures balanced partitions for parallel processing
- **Modulo-Based** offers deterministic, scalable distribution
- All strategies preserve data integrity (total records)
- Performance differences minimal for datasets <10K records

**Strategy recommendations:**
- **Use Value Groups** when: Business logic requires specific customer segments, manual group definition needed, relationship-based splitting
- **Use Equal Size** when: Load balancing required, parallel processing pipelines, fair distribution needed
- **Use Modulo** when: Distributed systems, database sharding, horizontal scaling, consistent hashing required

**Performance considerations:**
- For datasets >100K records: Consider using `use_dask=True` with modulo
- For value_groups with many groups: Use `use_vectorization=True` and `parallel_processes=4`
- Monitor memory usage with very large datasets

**Next steps:**
- Test with your production datasets
- Tune partition counts for optimal performance
- Implement in your data pipeline
- Monitor distribution balance in production

**Resources:**
- 📖 [PAMOLA Documentation](../../docs/)
- 🐙 [GitHub Repository](https://github.com/DGT-Network/PAMOLA)